# Yoga Pose Detection - Chair Pose (Utkatasana) Video Inference

This notebook runs trained pose-classification models on a video file.
It:
- Extracts MediaPipe landmarks from every Nth frame
- Engineers the exact same features used during training (from the pose YAML config)
- Handles **orientation-aware** joint resolution (`camera_*` semantic roles) so
  features always read from the camera-facing (visible) anatomical side
- Predicts the pose class + probability using a saved pipeline
- Displays the annotated video inline with skeleton overlay and prediction text

**Chair Pose classes:** `correct` · `shallow_squat` · `dropped_arms` · `bent_forward`

> Update only the paths in **Cell 1** - everything else is pose-agnostic.

---
## Cell 1 - Configuration (edit this cell only)

In [15]:
from pathlib import Path

#  Pose identity
POSE_NAME      = "chair_pose"
EXPERIMENT_TAG = "full_class"   # or "reduced_class" if a class was dropped

#  Saved artefact paths
# Root that contains models/ metadata/ results/ sub-folders
SAVED_FILES_ROOT = Path("../models/saved_files") / f"{POSE_NAME}_files"

DIR_MODELS   = SAVED_FILES_ROOT / "models"
DIR_METADATA = SAVED_FILES_ROOT / "metadata"
DIR_RESULTS  = SAVED_FILES_ROOT / "results"

#  Model selection
# Set to None  -> auto-selects the best model from the results CSV (recommended)
# Set to a string to force a specific model, e.g. "svc" or "xgboost"
FORCE_MODEL: str | None = None

AVAILABLE_MODELS = ["logistic_regression", "svc", "knn", "xgboost"]

#  YAML config (must be the same YAML used during feature engineering)
YAML_PATH = Path("../../configs/poses/chair_pose.yaml")

#  Confidence threshold
# Predictions below this confidence are shown as "unknown".
# Set to 0.0 to disable thresholding.
PROBABILITY_THRESHOLD = 0

#  Smoothing - sliding window majority vote
SMOOTHING_WINDOW = 15   # number of recent frames to vote over
MIN_HOLD_FRAMES  = 10   # minimum frames to hold a prediction before switching

#  Orientation detection
# Chair pose is recorded side-on (sagittal plane).
# Orientation is auto-detected per frame using a nose + shoulder consensus
# vs the hip centre heuristic (matching chair_specific_le.py):
#   right-facing: nose.x > hip_center.x AND shoulder_center.x > hip_center.x
#                 -> LEFT side of body faces camera (is_left_facing = False)
#   left-facing:  nose.x < hip_center.x AND shoulder_center.x < hip_center.x
#                 -> RIGHT side of body faces camera (is_left_facing = True)
# Set to None to auto-detect per frame (recommended).
# Set to True/False to force a specific orientation for the whole video.
FORCE_IS_LEFT_FACING: bool | None = None

#  Feature columns (must match the order used during training)
# Derived from chair_pose.yaml -> features_config -> engineered_features
FEATURE_COLUMNS = [
    "camera_knee_flexion_angle",
    "camera_torso_hip_knee_angle",
    "camera_shoulder_arm_angle",
    "camera_wrist_shoulder_vertical_delta",
    "camera_hip_knee_vertical_delta",
    "camera_shoulder_ankle_horizontal_offset",
    "camera_knee_ankle_horizontal_overhang",
]

#  Video
VIDEO_PATH = Path("test_videos/correct_8.mp4")   # relative to this notebook
FRAME_STEP = 1    # process every Nth frame (1 = every frame, 2 = every other)

#  Playback window
WINDOW_NAME  = f"{POSE_NAME} - Pose Detection"
# cv2.waitKey delay in ms between frames. Lower = faster playback.
# 1  -> as fast as inference allows (real-time feel on most machines)
# 33 -> ~30 fps cap
WAITKEY_MS   = 1
# Resize the display window to this width (height scales proportionally).
# Set to None to keep original video resolution.
DISPLAY_WIDTH: int | None = 960

#  MediaPipe model
# Will be auto-downloaded next to this notebook if not present
MEDIAPIPE_MODEL_PATH = Path("pose_landmarker_lite.task")
MEDIAPIPE_MODEL_URL  = (
    "https://storage.googleapis.com/mediapipe-models/"
    "pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
)

print("Configuration loaded.")
print(f"  Pose           : {POSE_NAME}")
print(f"  Experiment tag : {EXPERIMENT_TAG}")
print(f"  Saved files    : {SAVED_FILES_ROOT}  (exists={SAVED_FILES_ROOT.exists()})")
print(f"  YAML           : {YAML_PATH}  (exists={YAML_PATH.exists()})")
print(f"  Video          : {VIDEO_PATH}  (exists={VIDEO_PATH.exists()})")
print(f"  Force model    : {FORCE_MODEL or 'auto (best from results CSV)'}")
print(f"  Orientation    : {('auto-detect' if FORCE_IS_LEFT_FACING is None else ('left-facing' if FORCE_IS_LEFT_FACING else 'right-facing'))}")

Configuration loaded.
  Pose           : chair_pose
  Experiment tag : full_class
  Saved files    : ..\models\saved_files\chair_pose_files  (exists=True)
  YAML           : ..\..\configs\poses\chair_pose.yaml  (exists=True)
  Video          : test_videos\correct_8.mp4  (exists=True)
  Force model    : auto (best from results CSV)
  Orientation    : auto-detect


---
## Cell 2 - Imports

In [16]:
import base64
import math
import urllib.request
import warnings

import cv2
import joblib
import mediapipe as mp
import numpy as np
import pandas as pd
import yaml
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision import RunningMode

print("All imports OK.")

All imports OK.


---
## Cell 3 - Skeleton, colour & orientation constants

Chair Pose is recorded **side-on** (sagittal plane), so only the camera-facing
side is reliable. All features use `camera_*` semantic roles that resolve to
the visible (camera-facing) landmark index depending on which way the
practitioner faces. The highlighted (cyan-yellow) skeleton connections follow
this semantic mapping and are resolved per-frame at draw time.

In [17]:
# Full MediaPipe Pose skeleton connections (33 landmarks)
SKELETON_CONNECTIONS = [
    # Face
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10),
    # Torso
    (11, 12), (11, 23), (12, 24), (23, 24),
    # Left arm
    (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
    # Right arm
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
    # Left leg
    (23, 25), (25, 27), (27, 29), (27, 31), (29, 31),
    # Right leg
    (24, 26), (26, 28), (28, 30), (28, 32), (30, 32),
]

# MediaPipe landmark index constants
_LEFT_SHOULDER  = 11
_RIGHT_SHOULDER = 12
_LEFT_WRIST     = 15
_RIGHT_WRIST    = 16
_LEFT_HIP       = 23
_RIGHT_HIP      = 24
_LEFT_KNEE      = 25
_RIGHT_KNEE     = 26
_LEFT_ANKLE     = 27
_RIGHT_ANKLE    = 28

# Orientation map: semantic role -> (right-facing idx, left-facing idx)
#
# Side-view Chair Pose convention (matches chair_specific_fe.py):
#   right-facing (is_left=False): person faces RIGHT, LEFT side faces camera
#                                 -> camera_* roles resolve to LEFT landmarks
#   left-facing  (is_left=True) : person faces LEFT, RIGHT side faces camera
#                                 -> camera_* roles resolve to RIGHT landmarks
_ORIENTATION_MAP: dict[str, tuple[int, int]] = {
    "camera_shoulder": (_LEFT_SHOULDER, _RIGHT_SHOULDER),
    "camera_wrist":    (_LEFT_WRIST,    _RIGHT_WRIST),
    "camera_hip":      (_LEFT_HIP,      _RIGHT_HIP),
    "camera_knee":     (_LEFT_KNEE,     _RIGHT_KNEE),
    "camera_ankle":    (_LEFT_ANKLE,    _RIGHT_ANKLE),
}


def _resolve_joint(role_or_int, is_left: bool) -> int:
    """Resolve a semantic role string or raw int to a concrete MediaPipe index."""
    if isinstance(role_or_int, str):
        right_idx, left_idx = _ORIENTATION_MAP[role_or_int]
        return left_idx if is_left else right_idx
    return int(role_or_int)


def resolve_joints(joints: list, is_left: bool) -> list[int]:
    """Resolve an entire joint list for the given orientation."""
    return [_resolve_joint(j, is_left) for j in joints]


# Feature-relevant connections derived from chair_pose.yaml.
# Defined as semantic tuples; draw_skeleton resolves them per-frame.
# Each entry: (role_or_int_A, role_or_int_B)
FEATURE_CONNECTIONS_SEMANTIC = [
    # camera_knee_flexion_angle: camera_hip -> camera_knee -> camera_ankle
    ("camera_hip",      "camera_knee"),
    ("camera_knee",     "camera_ankle"),
    # camera_torso_hip_knee_angle: camera_shoulder -> camera_hip -> camera_knee
    ("camera_shoulder", "camera_hip"),
    # camera_shoulder_arm_angle: camera_hip -> camera_shoulder -> camera_wrist
    ("camera_shoulder", "camera_wrist"),
    # camera_shoulder_ankle_horizontal_offset
    ("camera_shoulder", "camera_ankle"),
    # camera_hip_knee_vertical_delta (already covered above, kept for clarity)
    # camera_knee_ankle_horizontal_overhang (already covered above)
]

# BGR colours
COLOUR_SKELETON = (200, 200, 200)   # light grey  - standard skeleton
COLOUR_FEATURE  = (0, 220, 255)     # vivid cyan-yellow - feature-relevant
COLOUR_LANDMARK = (255, 255, 255)   # white dots for all landmarks
COLOUR_FEAT_DOT = (0, 220, 255)     # matching dot for feature landmarks

# Semantic roles used for feature highlighting
_ORIENT_ROLES = [
    "camera_shoulder", "camera_wrist",
    "camera_hip",      "camera_knee",  "camera_ankle",
]

# No static (orientation-independent) feature ids for chair pose
_STATIC_FEATURE_IDS: set[int] = set()

print(f"Standard skeleton connections : {len(SKELETON_CONNECTIONS)}")
print(f"Semantic feature connections  : {len(FEATURE_CONNECTIONS_SEMANTIC)}")
print(f"Orientation roles             : {_ORIENT_ROLES}")

Standard skeleton connections : 35
Semantic feature connections  : 5
Orientation roles             : ['camera_shoulder', 'camera_wrist', 'camera_hip', 'camera_knee', 'camera_ankle']


---
## Cell 4 - Orientation detection helper

Chair Pose is a **side-view** pose, so orientation detection uses a nose +
shoulder consensus vs the hip centre (matching `chair_specific_le.py`):

- `nose.x > hip_center.x` **AND** `shoulder_center.x > hip_center.x`
  → person faces **right** → LEFT side faces camera (`is_left_facing = False`)
- `nose.x < hip_center.x` **AND** `shoulder_center.x < hip_center.x`
  → person faces **left**  → RIGHT side faces camera (`is_left_facing = True`)
- Signals disagree or visibility is low → fall back to dataset default
  (`right-facing` / `is_left_facing = False`).

This exactly mirrors the training-time `is_left_facing` flag.

In [18]:
VISIBILITY_THRESHOLD   = 0.5
DEFAULT_IS_LEFT_FACING = False   # dataset default: right-facing


def detect_is_left_facing(landmarks) -> bool:
    """
    Determine whether the subject is left-facing in the frame.

    Strategy: nose + shoulder consensus vs hip centre (matches chair_specific_le.py).

        nose.x > hip_center.x  AND  shoulder_center.x > hip_center.x
            -> person faces RIGHT  ->  is_left_facing = False  (default)

        nose.x < hip_center.x  AND  shoulder_center.x < hip_center.x
            -> person faces LEFT   ->  is_left_facing = True

    If signals disagree or any key landmark is below VISIBILITY_THRESHOLD,
    falls back to DEFAULT_IS_LEFT_FACING (False = right-facing).

    Args:
        landmarks: pose_landmarks list from a MediaPipe detection result.

    Returns:
        True  -> left-facing  (RIGHT side of body faces camera)
        False -> right-facing (LEFT  side of body faces camera, dataset default)
    """
    nose       = landmarks[0]
    l_shoulder = landmarks[11]
    r_shoulder = landmarks[12]
    l_hip      = landmarks[23]
    r_hip      = landmarks[24]

    # Visibility guard
    key_lms = [nose, l_shoulder, r_shoulder, l_hip, r_hip]
    if any(lm.visibility < VISIBILITY_THRESHOLD for lm in key_lms):
        return DEFAULT_IS_LEFT_FACING

    hip_center_x      = (l_hip.x      + r_hip.x)      / 2.0
    shoulder_center_x = (l_shoulder.x + r_shoulder.x) / 2.0

    nose_ahead     = nose.x            > hip_center_x
    shoulder_ahead = shoulder_center_x  > hip_center_x

    if nose_ahead and shoulder_ahead:
        return False   # both agree -> right-facing
    elif not nose_ahead and not shoulder_ahead:
        return True    # both agree -> left-facing
    else:
        return DEFAULT_IS_LEFT_FACING   # signals disagree -> use default


print("Orientation detection helper defined.")

Orientation detection helper defined.


---
## Cell 5 - Feature engineering helpers

These functions are a **direct scalar port** of the training-time feature
engineering pipeline (`chair_specific_fe.py`). They operate on a single frame
rather than a DataFrame so they are efficient for online inference.

Key characteristics of Chair Pose features:
- Joint resolution goes through `_ORIENTATION_MAP` using `camera_*` semantic
  roles so features always read from the camera-facing (visible) side.
- `camera_knee_flexion_angle` uses `mode: 2d` (X/Y only, no Z depth jitter).
- `camera_torso_hip_knee_angle` and `camera_shoulder_arm_angle` use 3-D.
- All distance/offset features are normalised by torso length
  (`camera_shoulder → camera_hip`).

In [19]:
EPSILON = 1e-6


#  Low-level coordinate helpers

def _xy(landmarks, idx: int) -> np.ndarray:
    """Return (x, y) for landmark idx as a 1-D float64 array."""
    lm = landmarks[idx]
    return np.array([lm.x, lm.y], dtype=np.float64)


def _xyz(landmarks, idx: int) -> np.ndarray:
    """Return (x, y, z) for landmark idx as a 1-D float64 array."""
    lm = landmarks[idx]
    return np.array([lm.x, lm.y, lm.z], dtype=np.float64)


def _x(landmarks, idx: int) -> float:
    """Return the raw X coordinate for landmark idx."""
    return float(landmarks[idx].x)


def _y(landmarks, idx: int) -> float:
    """Return the raw Y coordinate for landmark idx."""
    return float(landmarks[idx].y)


#  Geometry primitives

def _angle_at_vertex_3d(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    """
    3-D angle at vertex B formed by rays B->A and B->C, in degrees.
    Used when the YAML config does NOT specify mode: 2d.
    """
    ba = a - b
    bc = c - b
    cosine = np.clip(
        np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + EPSILON),
        -1.0, 1.0,
    )
    return float(np.degrees(np.arccos(cosine)))


def _angle_at_vertex_2d(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    """
    2-D screen-plane angle at vertex B (X/Y only), in degrees.
    Used when the YAML config specifies mode: 2d.
    Excluding Z prevents depth-jitter from distorting knee angles.
    """
    ba = a - b
    bc = c - b
    cosine = np.clip(
        np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + EPSILON),
        -1.0, 1.0,
    )
    return float(np.degrees(np.arccos(cosine)))


def _euclidean_2d(a: np.ndarray, b: np.ndarray) -> float:
    """2-D Euclidean distance between two (x, y) points."""
    return float(np.linalg.norm(a - b))


#  Per-feature-type compute functions

def _compute_joint_angle(
    landmarks, cfg: dict, is_left: bool
) -> float:
    """
    Compute a single joint-angle feature with orientation-aware joint resolution.
    Uses 2-D maths when cfg contains ``mode: '2d'``; falls back to 3-D otherwise.
    """
    j = resolve_joints(cfg["joints"], is_left)
    if cfg.get("mode") == "2d":
        return _angle_at_vertex_2d(
            _xy(landmarks, j[0]),
            _xy(landmarks, j[1]),
            _xy(landmarks, j[2]),
        )
    return _angle_at_vertex_3d(
        _xyz(landmarks, j[0]),
        _xyz(landmarks, j[1]),
        _xyz(landmarks, j[2]),
    )


def _compute_distance_or_offset(
    landmarks, cfg: dict, is_left: bool
) -> float:
    """
    Compute a single spatial-distance or alignment-offset feature.

    Chair Pose uses two compute types (matching chair_specific_fe.py):
        vertical_delta  - |y_b - y_a|  (wrist height, hip-knee drop)
        lateral_delta   - |x_a - x_b|  (shoulder-ankle, knee-ankle overhang)

    All distance features are optionally normalised by torso length
    (camera_shoulder -> camera_hip).
    """
    name    = cfg["name"]
    compute = cfg.get("compute", "").strip().lower()
    j       = resolve_joints(cfg["joints"], is_left)

    try:
        if compute == "vertical_delta":
            # |y_b - y_a|  (MediaPipe Y increases downward)
            val = abs(_y(landmarks, j[1]) - _y(landmarks, j[0]))

        elif compute == "lateral_delta":
            # |x_a - x_b|  - horizontal separation on screen
            val = abs(_x(landmarks, j[0]) - _x(landmarks, j[1]))

        else:
            # Euclidean 2-D distance (fallback for plain spatial_distances)
            val = _euclidean_2d(_xy(landmarks, j[0]), _xy(landmarks, j[1]))

    except Exception as exc:
        warnings.warn(f"Failed computing feature '{name}': {exc}")
        return float("nan")

    if "normalization_factor" in cfg:
        nf    = resolve_joints(cfg["normalization_factor"], is_left)
        scale = _euclidean_2d(_xy(landmarks, nf[0]), _xy(landmarks, nf[1]))
        val   = val / (scale + EPSILON)

    return float(val)


#  Master per-frame feature extractor

def extract_features_from_landmarks(
    landmarks,
    feat_config: dict,
    feature_columns: list[str],
    is_left: bool,
) -> np.ndarray | None:
    """
    Compute all engineered features for a single Chair Pose frame.

    Args:
        landmarks      : pose_landmarks list from MediaPipe.
        feat_config    : the ``engineered_features`` block from the pose YAML.
        feature_columns: ordered feature names matching training order.
        is_left        : orientation flag for this frame.

    Returns:
        1-D numpy float64 array of shape (n_features,), or None if any value
        is NaN (pose not usable for inference).
    """
    feature_map: dict[str, float] = {}

    # Joint angles (uses 2d or 3d as specified per feature in YAML)
    for cfg in feat_config.get("joint_angles", []):
        feature_map[cfg["name"]] = _compute_joint_angle(landmarks, cfg, is_left)

    # Spatial distances (vertical_delta / lateral_delta)
    for cfg in feat_config.get("spatial_distances", []):
        feature_map[cfg["name"]] = _compute_distance_or_offset(landmarks, cfg, is_left)

    # Alignment offsets (lateral_delta / vertical_delta)
    for cfg in feat_config.get("alignment_offsets", []):
        feature_map[cfg["name"]] = _compute_distance_or_offset(landmarks, cfg, is_left)

    try:
        row = np.array([feature_map[col] for col in feature_columns], dtype=np.float64)
    except KeyError as e:
        raise KeyError(
            f"Feature {e} is in FEATURE_COLUMNS but was not computed. "
            f"Check that the YAML config covers all required features."
        ) from e

    if np.any(np.isnan(row)):
        return None

    return row


print("Feature engineering helpers defined.")

Feature engineering helpers defined.


---
## Cell 6 - Drawing helpers

The skeleton drawing resolves semantic `camera_*` joint roles per-frame so
the highlighted connections always track the correct camera-facing side
regardless of which way the practitioner faces.

In [20]:
def _landmark_px(lm, frame_w: int, frame_h: int) -> tuple[int, int]:
    """Convert normalised MediaPipe coordinates to pixel coordinates."""
    return int(lm.x * frame_w), int(lm.y * frame_h)


def draw_skeleton(
    frame: np.ndarray,
    landmarks,
    is_left: bool,
) -> np.ndarray:
    """
    Draw the full pose skeleton on ``frame``.

    - Standard connections are drawn in light grey.
    - Feature-relevant connections are drawn in vivid cyan-yellow on top.
      Semantic camera_* roles are resolved for the current orientation.
    - All landmark dots are white; feature-relevant ones match the feature colour.

    Args:
        frame     : BGR frame (modified in-place and returned).
        landmarks : pose_landmarks list from MediaPipe.
        is_left   : orientation flag - True = left-facing.

    Returns:
        Annotated frame.
    """
    h, w = frame.shape[:2]
    px   = [_landmark_px(lm, w, h) for lm in landmarks]

    # Resolve semantic feature connections to concrete index pairs for this frame
    resolved_feat_conns = set()
    for a, b in FEATURE_CONNECTIONS_SEMANTIC:
        ia = _resolve_joint(a, is_left)
        ib = _resolve_joint(b, is_left)
        resolved_feat_conns.add((ia, ib))
        resolved_feat_conns.add((ib, ia))   # undirected

    # Feature-relevant landmark ids for this orientation
    feat_ids = set(_STATIC_FEATURE_IDS)
    for role in _ORIENT_ROLES:
        feat_ids.add(_resolve_joint(role, is_left))

    # Draw standard skeleton (background)
    for a, b in SKELETON_CONNECTIONS:
        if (a, b) not in resolved_feat_conns:
            cv2.line(frame, px[a], px[b], COLOUR_SKELETON, 2, cv2.LINE_AA)

    # Draw feature-relevant connections (foreground)
    for a, b in FEATURE_CONNECTIONS_SEMANTIC:
        ia = _resolve_joint(a, is_left)
        ib = _resolve_joint(b, is_left)
        cv2.line(frame, px[ia], px[ib], COLOUR_FEATURE, 3, cv2.LINE_AA)

    # Draw landmark dots
    for i, pos in enumerate(px):
        colour = COLOUR_FEAT_DOT if i in feat_ids else COLOUR_LANDMARK
        radius = 5 if i in feat_ids else 3
        cv2.circle(frame, pos, radius, colour, -1, cv2.LINE_AA)

    return frame


def draw_prediction_overlay(
    frame: np.ndarray,
    pose_name: str,
    predicted_class: str,
    probability: float | None,
    feedback: str,
    frame_idx: int,
    orientation_label: str,
) -> np.ndarray:
    """
    Render a semi-transparent info panel in the top-left corner with:
        - Pose name
        - Detected orientation (left-facing / right-facing)
        - Predicted class
        - Prediction probability (if available)
        - Corrective feedback from the YAML
        - Current frame index

    Args:
        frame             : BGR frame (modified in-place and returned).
        pose_name         : e.g. 'chair_pose'
        predicted_class   : e.g. 'correct' or 'shallow_squat'
        probability       : float in [0, 1] or None
        feedback          : corrective text from the YAML feedback block
        frame_idx         : frame number shown in the corner
        orientation_label : 'right-facing' or 'left-facing'

    Returns:
        Annotated frame.
    """
    font        = cv2.FONT_HERSHEY_SIMPLEX
    pad         = 10
    line_h      = 28
    small_scale = 0.55
    large_scale = 0.75

    prob_str = f"{probability * 100:.1f}%" if probability is not None else "N/A"

    is_correct    = predicted_class.lower() == "correct"
    banner_colour = (30, 160, 30) if is_correct else (0, 130, 200)

    lines = [
        (f"Pose  : {pose_name.replace('_', ' ').title()}", small_scale, (220, 220, 220)),
        (f"Orient: {orientation_label}",                   small_scale, (180, 200, 255)),
        (f"Class : {predicted_class.replace('_', ' ').upper()}", large_scale, (255, 255, 255)),
        (f"Prob  : {prob_str}",                            small_scale, (200, 220, 255)),
        (f"Frame : {frame_idx}",                           small_scale, (180, 180, 180)),
    ]

    if feedback:
        words, current = feedback.split(), ""
        wrap_lines = []
        for word in words:
            if len(current) + len(word) + 1 > 45:
                wrap_lines.append(current.strip())
                current = word + " "
            else:
                current += word + " "
        if current:
            wrap_lines.append(current.strip())
        for wl in wrap_lines:
            lines.append((wl, small_scale - 0.05, (160, 230, 160)))

    panel_h = pad * 2 + len(lines) * line_h
    panel_w = 360

    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (panel_w, panel_h), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)

    cv2.rectangle(frame, (0, 0), (5, panel_h), banner_colour, -1)

    for row_i, (text, scale, colour) in enumerate(lines):
        y = pad + row_i * line_h + line_h // 2
        cv2.putText(frame, text, (14, y), font, scale, colour, 1, cv2.LINE_AA)

    return frame


def frame_to_jpeg_b64(frame: np.ndarray, quality: int = 85) -> str:
    """Encode a BGR frame to a base-64 JPEG string for inline HTML display."""
    ok, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        raise RuntimeError("cv2.imencode failed.")
    return base64.b64encode(buf.tobytes()).decode()


print("Drawing helpers defined.")

Drawing helpers defined.


---
## Cell 7 - Load artefacts

In [21]:
#  1. Load YAML config
assert YAML_PATH.exists(), f"YAML not found: {YAML_PATH}"
with open(YAML_PATH, "r") as f:
    pose_yaml = yaml.safe_load(f)

feat_config = pose_yaml.get("features_config", {}).get("engineered_features", {})
assert feat_config, "No 'features_config.engineered_features' block found in YAML."

yaml_feedback: dict = pose_yaml.get("feedback", {})
print(f"YAML loaded. Feedback classes: {list(yaml_feedback.keys())}")

#  2. Select model
if FORCE_MODEL is not None:
    assert FORCE_MODEL in AVAILABLE_MODELS, (
        f"FORCE_MODEL='{FORCE_MODEL}' is not in {AVAILABLE_MODELS}"
    )
    selected_model = FORCE_MODEL
    print(f"Model forced to: {selected_model}")
else:
    results_csv = DIR_RESULTS / f"{POSE_NAME}_{EXPERIMENT_TAG}_experiment_results.csv"
    assert results_csv.exists(), (
        f"Results CSV not found: {results_csv}\n"
        f"Set FORCE_MODEL to specify a model manually."
    )
    results_df     = pd.read_csv(results_csv)
    best_row       = results_df.sort_values("test_f1_weighted", ascending=False).iloc[0]
    selected_model = best_row["model"]
    print(f"Results CSV loaded. Best model by F1: {selected_model} "
          f"(test_f1={best_row['test_f1_weighted']:.4f})")

#  3. Load pipeline and label encoder
prefix        = f"{POSE_NAME}_{EXPERIMENT_TAG}_{selected_model}"
pipeline_path = DIR_MODELS / f"{prefix}_pipeline.joblib"
encoder_path  = DIR_MODELS / f"{prefix}_label_encoder.joblib"

assert pipeline_path.exists(), f"Pipeline not found: {pipeline_path}"
assert encoder_path.exists(),  f"Label encoder not found: {encoder_path}"

pipeline      = joblib.load(pipeline_path)
label_encoder = joblib.load(encoder_path)

print(f"Pipeline loaded   : {pipeline_path.name}")
print(f"Label encoder     : {encoder_path.name}")
print(f"Known classes     : {list(label_encoder.classes_)}")

#  4. Check probability support
HAS_PROBA = hasattr(pipeline, "predict_proba")
print(f"Probability support: {HAS_PROBA}")

YAML loaded. Feedback classes: ['correct', 'shallow_squat', 'dropped_arms', 'bent_forward']
Results CSV loaded. Best model by F1: svc (test_f1=0.8820)
Pipeline loaded   : chair_pose_full_class_svc_pipeline.joblib
Label encoder     : chair_pose_full_class_svc_label_encoder.joblib
Known classes     : ['bent_forward', 'correct', 'dropped_arms', 'shallow_squat']
Probability support: True


---
## Cell 8 - Ensure MediaPipe model file

In [22]:
def ensure_mediapipe_model(model_path: Path, url: str) -> None:
    """Download the MediaPipe .task model file if it is not already present."""
    if model_path.exists():
        print(f"MediaPipe model found: {model_path}")
        return
    print(f"Downloading MediaPipe model from:\n  {url}")
    urllib.request.urlretrieve(url, model_path)
    print(f"Saved to: {model_path}")


ensure_mediapipe_model(MEDIAPIPE_MODEL_PATH, MEDIAPIPE_MODEL_URL)
assert VIDEO_PATH.exists(), f"Video file not found: {VIDEO_PATH}"

MediaPipe model found: pose_landmarker_lite.task


---
## Cell 9 - Stream inference + display in a cv2 window

Detect → annotate → display in one tight loop - no frame buffering.
A `cv2` popup window opens automatically.

**Controls while the window is open:**
- Press **Q** or **Esc** to quit early
- Press **Space** to pause / resume

The kernel cell completes when the video ends or you quit.

**How to tune it (in Cell 1)**

| Scenario | Adjustment |
|---|---|
| Still flickering | Increase `SMOOTHING_WINDOW` (try `20–30`) |
| Too slow to react to a real pose change | Decrease `MIN_HOLD_FRAMES` (try `5–8`) |
| `"unknown"` flashing briefly during transitions | Increase `MIN_HOLD_FRAMES` |
| Testing a static video | `SMOOTHING_WINDOW=10`, `MIN_HOLD_FRAMES=8` is a good baseline |
| Subject facing left the whole time | Set `FORCE_IS_LEFT_FACING = True` in Cell 1 |
| Subject facing right the whole time | Set `FORCE_IS_LEFT_FACING = False` in Cell 1 |

In [23]:
from collections import deque


class PredictionSmoother:
    """
    Majority-vote smoother over a sliding window of recent predictions.
    A new class only takes over when it wins the vote AND the current
    class has been held for at least min_hold_frames.
    """
    def __init__(self, window_size: int, min_hold_frames: int):
        self.window        = deque(maxlen=window_size)
        self.min_hold      = min_hold_frames
        self.current_class = None
        self.hold_count    = 0

    def update(self, new_class: str) -> str:
        self.window.append(new_class)
        vote_winner = max(set(self.window), key=self.window.count)

        if self.current_class is None:
            self.current_class = vote_winner
            self.hold_count    = 1
        elif vote_winner == self.current_class:
            self.hold_count += 1
        elif self.hold_count >= self.min_hold:
            self.current_class = vote_winner
            self.hold_count    = 1
        else:
            self.hold_count += 1

        return self.current_class

In [24]:
def stream_inference_to_window(
    video_path: Path,
    mediapipe_model_path: Path,
    pipeline,
    label_encoder,
    feat_config: dict,
    feature_columns: list[str],
    yaml_feedback: dict,
    pose_name: str,
    window_name: str,
    frame_step: int = 1,
    waitkey_ms: int = 1,
    display_width: int | None = None,
    force_is_left: bool | None = None,
) -> dict:
    """
    Stream a Chair Pose video through the full inference pipeline and display
    it live in a cv2 popup window. Nothing is buffered - each frame is
    processed and shown immediately.

    Chair-Pose-specific additions:
        - Per-frame orientation detection (right-facing vs left-facing)
          using nose + shoulder consensus vs hip centre.
        - camera_* orientation-aware feature extraction and skeleton highlighting.
        - Orientation label shown in the HUD overlay.

    Keyboard controls inside the window:
        Q / Esc   quit
        Space     pause / resume

    Args:
        force_is_left : None = auto-detect per frame;
                        True = always left-facing (RIGHT side faces camera);
                        False = always right-facing (LEFT side faces camera).

    Returns:
        Summary dict with processed frame count and per-class prediction counts.
    """
    options = mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(
            model_asset_path=str(mediapipe_model_path)
        ),
        running_mode=RunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
    )

    cap = cv2.VideoCapture(str(video_path))
    assert cap.isOpened(), f"Cannot open video: {video_path}"

    video_fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video   : {video_path.name}")
    print(f"FPS     : {video_fps:.1f}  |  Total frames : {total_frames}")
    print(f"Window  : '{window_name}'  - press Q or Esc to quit, Space to pause")

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    if display_width is not None:
        orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        scale  = display_width / max(orig_w, 1)
        cv2.resizeWindow(window_name, display_width, int(orig_h * scale))

    frame_count     = 0
    processed_count = 0
    no_pose_count   = 0
    pred_counts: dict[str, int] = {}
    paused   = False
    smoother = PredictionSmoother(SMOOTHING_WINDOW, MIN_HOLD_FRAMES)

    with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            if paused:
                key = cv2.waitKey(50) & 0xFF
                if key == ord(" "):
                    paused = False
                elif key in (ord("q"), 27):
                    print("Quit by user.")
                    break
                continue

            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            if (frame_count - 1) % frame_step != 0:
                continue

            processed_count += 1
            annotated    = frame.copy()
            timestamp_ms = int((frame_count / video_fps) * 1000)

            img_rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
            result   = landmarker.detect_for_video(mp_image, timestamp_ms)

            if not result.pose_landmarks:
                no_pose_count += 1
                cv2.putText(
                    annotated, "No pose detected",
                    (14, 50), cv2.FONT_HERSHEY_SIMPLEX,
                    1.0, (0, 60, 220), 2, cv2.LINE_AA,
                )
            else:
                landmarks = result.pose_landmarks[0]

                #  Orientation detection
                is_left = (
                    detect_is_left_facing(landmarks)
                    if force_is_left is None
                    else force_is_left
                )
                orientation_label = "left-facing" if is_left else "right-facing"

                #  Skeleton
                draw_skeleton(annotated, landmarks, is_left)

                #  Feature engineering
                feature_vec = extract_features_from_landmarks(
                    landmarks, feat_config, feature_columns, is_left
                )

                if feature_vec is None:
                    cv2.putText(
                        annotated, "Degenerate pose",
                        (14, 50), cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (0, 100, 220), 2, cv2.LINE_AA,
                    )
                else:
                    #  Inference
                    X_frame    = feature_vec.reshape(1, -1)
                    pred_enc   = pipeline.predict(X_frame)[0]
                    pred_class = label_encoder.inverse_transform([pred_enc])[0]

                    probability = None
                    if HAS_PROBA:
                        proba_vec   = pipeline.predict_proba(X_frame)[0]
                        probability = float(proba_vec[pred_enc])

                    # Apply confidence threshold
                    if probability is not None and probability < PROBABILITY_THRESHOLD:
                        raw_class = "unknown"
                    else:
                        raw_class = pred_class

                    # Smooth over sliding window
                    display_class    = smoother.update(raw_class)
                    display_feedback = yaml_feedback.get(
                        display_class, "Pose unclear - please adjust your position."
                    )

                    pred_counts[display_class] = pred_counts.get(display_class, 0) + 1

                    draw_prediction_overlay(
                        annotated,
                        pose_name=pose_name,
                        predicted_class=display_class,
                        probability=probability,
                        feedback=display_feedback,
                        frame_idx=frame_count,
                        orientation_label=orientation_label,
                    )

            cv2.imshow(window_name, annotated)

            key = cv2.waitKey(waitkey_ms) & 0xFF
            if key in (ord("q"), 27):
                print("Quit by user.")
                break
            if key == ord(" "):
                paused = True

    cap.release()
    cv2.destroyWindow(window_name)

    inferrable = max(1, processed_count - no_pose_count)
    print(f"\nDone. Processed {processed_count} frames | No-pose: {no_pose_count}")
    print("Prediction distribution:")
    for cls, cnt in sorted(pred_counts.items(), key=lambda x: -x[1]):
        print(f"  {cls:<25} {cnt:>5} frames  ({cnt / inferrable * 100:.1f}%)")

    return {
        "processed_frames": processed_count,
        "no_pose_frames"  : no_pose_count,
        "pred_counts"     : pred_counts,
    }


summary = stream_inference_to_window(
    video_path           = VIDEO_PATH,
    mediapipe_model_path = MEDIAPIPE_MODEL_PATH,
    pipeline             = pipeline,
    label_encoder        = label_encoder,
    feat_config          = feat_config,
    feature_columns      = FEATURE_COLUMNS,
    yaml_feedback        = yaml_feedback,
    pose_name            = POSE_NAME,
    window_name          = WINDOW_NAME,
    frame_step           = FRAME_STEP,
    waitkey_ms           = WAITKEY_MS,
    display_width        = DISPLAY_WIDTH,
    force_is_left        = FORCE_IS_LEFT_FACING,
)

Video   : correct_8.mp4
FPS     : 30.0  |  Total frames : 83
Window  : 'chair_pose - Pose Detection'  - press Q or Esc to quit, Space to pause

Done. Processed 83 frames | No-pose: 0
Prediction distribution:
  correct                      83 frames  (100.0%)


---
## Cell 10 - Re-run on a different video (optional)

Reuse the loaded pipeline without reloading anything - just point at a new video.

In [28]:
# Change this path and run this cell to test another video without reloading artefacts
ANOTHER_VIDEO = Path("test_videos/shallow_squat_1.mp4")

if ANOTHER_VIDEO.exists():
    stream_inference_to_window(
        video_path           = ANOTHER_VIDEO,
        mediapipe_model_path = MEDIAPIPE_MODEL_PATH,
        pipeline             = pipeline,
        label_encoder        = label_encoder,
        feat_config          = feat_config,
        feature_columns      = FEATURE_COLUMNS,
        yaml_feedback        = yaml_feedback,
        pose_name            = POSE_NAME,
        window_name          = WINDOW_NAME,
        frame_step           = FRAME_STEP,
        waitkey_ms           = WAITKEY_MS,
        display_width        = DISPLAY_WIDTH,
        force_is_left        = FORCE_IS_LEFT_FACING,
    )
else:
    print(f"File not found: {ANOTHER_VIDEO} - update the path above.")

Video   : shallow_squat_1.mp4
FPS     : 30.0  |  Total frames : 901
Window  : 'chair_pose - Pose Detection'  - press Q or Esc to quit, Space to pause
Quit by user.

Done. Processed 676 frames | No-pose: 0
Prediction distribution:
  shallow_squat               666 frames  (98.5%)
  dropped_arms                 10 frames  (1.5%)


---
## Cell 11 - Per-frame prediction summary (optional)

Tabular view of every processed frame - useful for spotting transitions,
debugging individual frames, or verifying orientation detection consistency.

In [ ]:
def build_prediction_log(
    video_path: Path,
    mediapipe_model_path: Path,
    pipeline,
    label_encoder,
    feat_config: dict,
    feature_columns: list[str],
    frame_step: int = 1,
    force_is_left: bool | None = None,
) -> pd.DataFrame:
    """
    Replay the video and collect per-frame predictions into a DataFrame.
    Cheaper to run separately (no drawing overhead) for analysis purposes.

    Returns a DataFrame with columns:
        frame_number, is_left_facing, predicted_class, probability, <feature values...>
    """
    options = mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(
            model_asset_path=str(mediapipe_model_path)
        ),
        running_mode=RunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
    )

    cap         = cv2.VideoCapture(str(video_path))
    fps         = cap.get(cv2.CAP_PROP_FPS) or 30.0
    rows        = []
    frame_count = 0

    with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            if (frame_count - 1) % frame_step != 0:
                continue

            timestamp_ms = int((frame_count / fps) * 1000)
            img_rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image     = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
            result       = landmarker.detect_for_video(mp_image, timestamp_ms)

            row = {"frame_number": frame_count}

            if not result.pose_landmarks:
                row["is_left_facing"]  = None
                row["predicted_class"] = "no_pose"
                row["probability"]     = float("nan")
                rows.append(row)
                continue

            landmarks = result.pose_landmarks[0]
            is_left   = (
                detect_is_left_facing(landmarks)
                if force_is_left is None
                else force_is_left
            )
            row["is_left_facing"] = is_left

            feature_vec = extract_features_from_landmarks(
                landmarks, feat_config, feature_columns, is_left
            )

            if feature_vec is None:
                row["predicted_class"] = "degenerate_pose"
                row["probability"]     = float("nan")
                rows.append(row)
                continue

            for col, val in zip(feature_columns, feature_vec):
                row[col] = round(val, 6)

            X_frame    = feature_vec.reshape(1, -1)
            pred_enc   = pipeline.predict(X_frame)[0]
            pred_class = label_encoder.inverse_transform([pred_enc])[0]
            row["predicted_class"] = pred_class

            if HAS_PROBA:
                proba_vec          = pipeline.predict_proba(X_frame)[0]
                row["probability"] = round(float(proba_vec[pred_enc]), 4)
            else:
                row["probability"] = float("nan")

            rows.append(row)

    cap.release()
    return pd.DataFrame(rows)


log_df = build_prediction_log(
    video_path           = VIDEO_PATH,
    mediapipe_model_path = MEDIAPIPE_MODEL_PATH,
    pipeline             = pipeline,
    label_encoder        = label_encoder,
    feat_config          = feat_config,
    feature_columns      = FEATURE_COLUMNS,
    frame_step           = FRAME_STEP,
    force_is_left        = FORCE_IS_LEFT_FACING,
)

print(f"Per-frame log: {len(log_df)} rows")
print("\nClass distribution across frames:")
print(log_df["predicted_class"].value_counts().to_string())
print("\nOrientation distribution:")
print(log_df["is_left_facing"].value_counts().rename({True: 'left-facing', False: 'right-facing'}).to_string())
print()
log_df.head(20)